Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!sudo apt-get update
!sudo apt-get install -y bcftools tabix

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,913 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,772 kB]
Get:14

In [3]:
!pip install cyvcf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.3 MB/s eta 0:00:00


Import Packages

In [4]:
import os
import pandas as pd
from cyvcf2 import VCF
import joblib
import numpy as np

Directory Setup

In [10]:
BASE = "/content/drive/MyDrive/Bioinformatics/"

DATA_DIR = BASE + "data/"
PATIENT_DIR = DATA_DIR + "external/patient_vcfs/"
SNPEFF_DIR = BASE + "tools/snpEff/"
PROC_DIR      = DATA_DIR + "processed/"

# CHUNK_DIR     = PROC_DIR + "chunks/"
# ANN_DIR       = PROC_DIR + "annotated_chunks/"

MODEL_DIR = BASE + "models/"
OUT_DIR = BASE + "outputs/"
VARIANT_DIR = OUT_DIR + "variants/"

# os.makedirs(CHUNK_DIR, exist_ok=True)
# os.makedirs(ANN_DIR, exist_ok=True)
os.makedirs(VARIANT_DIR, exist_ok=True)

MODEL_PATH = MODEL_DIR + "rf_ann_model.pkl"

Patient Input File

In [12]:
RAW_VCF_DRIVE = PATIENT_DIR + "patient_rett.vcf"

Upload & Load Raw Patient VCF

In [11]:
'''
from google.colab import files
uploaded = files.upload()

for fname in uploaded.keys():
    PATIENT_VCF = PATIENT_DIR + fname
    os.rename(fname, PATIENT_VCF)

print("Uploaded patient VCF:", PATIENT_VCF)
'''

Saving sample1.vcf to sample1 (2).vcf


OSError: [Errno 18] Invalid cross-device link: 'sample1 (2).vcf' -> '/content/drive/MyDrive/Bioinformatics/data/external/patient_vcfs/sample1 (2).vcf'

Copy INPUT file to local disk

In [13]:
LOCAL_RAW_VCF = "/content/patient_rett.vcf"

!cp "$RAW_VCF_DRIVE" "$LOCAL_RAW_VCF"

print("Local copy ready:", LOCAL_RAW_VCF)

Local copy ready: /content/patient_rett.vcf


Normalize → compress → index using local disk

In [14]:
LOCAL_NORM_VCF = "/content/patient_rett.norm.vcf.gz"

!bcftools norm -m -any "$LOCAL_RAW_VCF" | bcftools sort -Oz -o "$LOCAL_NORM_VCF"
!tabix -p vcf "$LOCAL_NORM_VCF"

print("Normalized VCF ready:", LOCAL_NORM_VCF)

Writing to /tmp/bcftools.yJ14la
[W::vcf_parse] Contig 'X' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse_info] INFO 'ANN' is not defined in the header, assuming Type=String
[W::vcf_parse] Contig '16' is not defined in the header. (Quick workaround: index the file with tabix.)
[E::vcf_format] Invalid BCF, CONTIG id=0 not present in the header
[flush_buffer] Error: cannot write to -
[E::bcf_hdr_read] Input is not detected as bcf or vcf format
Could not read VCF/BCF headers from -
Cleaning
tbx_index_build failed: /content/patient_rett.norm.vcf.gz
Normalized VCF ready: /content/patient_rett.norm.vcf.gz


Run SnpEff annotation

In [15]:
# annotate to local plain VCF
LOCAL_ANN_VCF = "/content/patient_rett_annotated.vcf"

cmd = f'java -Xmx6g -jar "{SNPEFF_DIR}snpEff.jar" GRCh38.86 "{LOCAL_NORM_VCF}" > "{LOCAL_ANN_VCF}"'
print(cmd)
os.system(cmd)

print("Annotation complete:", LOCAL_ANN_VCF)

java -Xmx6g -jar "/content/drive/MyDrive/Bioinformatics/tools/snpEff/snpEff.jar" GRCh38.86 "/content/patient_rett.norm.vcf.gz" > "/content/patient_rett_annotated.vcf"
Annotation complete: /content/patient_rett_annotated.vcf


Compress + Index Annotated VCF

In [16]:
!bgzip -f "$LOCAL_ANN_VCF"   # creates patient1_annotated.vcf.gz
!tabix -p vcf "$LOCAL_ANN_VCF.gz"

LOCAL_ANN_VCF_GZ = LOCAL_ANN_VCF + ".gz"

print("Compressed + indexed:", LOCAL_ANN_VCF_GZ)

[E::hts_open_format] Failed to open file ".gz" : No such file or directory
Couldn't open ".gz": No such file or directory
Compressed + indexed: /content/patient_rett_annotated.vcf.gz


Parse annotated VCF.GZ

In [17]:
vcf = VCF(LOCAL_ANN_VCF_GZ)
rows = []

for var in vcf:
    ann = var.INFO.get("ANN")

    effect = impact = gene = biotype = None

    if ann:
        first = ann.split(",")[0]
        p = first.split("|")
        effect  = p[1] if len(p) > 1 else None
        impact  = p[2] if len(p) > 2 else None
        gene    = p[3] if len(p) > 3 else None
        biotype = p[7] if len(p) > 7 else None

    rows.append({
        "CHROM": var.CHROM,
        "POS": var.POS,
        "REF": var.REF,
        "ALT": var.ALT[0] if len(var.ALT) else "",
        "GeneName": gene,
        "Effect": effect,
        "Impact": impact,
        "Transcript_BioType": biotype
    })

df = pd.DataFrame(rows)
print("Parsed variants:", len(df))
df.head()

OSError: /content/patient_rett_annotated.vcf.gz is not valid bcf or vcf (format: 15 mode: r)

SAVE ONLY CSV (annotation) TO DRIVE

In [17]:
FINAL_ANNOTATED_CSV = "/content/drive/MyDrive/Bioinformatics/data/processed/patient1_annotated.csv"
df.to_csv(FINAL_ANNOTATED_CSV, index=False)

print("Saved annotation CSV:", FINAL_ANNOTATED_CSV)

Saved annotation CSV: /content/drive/MyDrive/Bioinformatics/data/processed/patient1_annotated.csv


ML Features

In [18]:
df["REF_len"] = df["REF"].apply(len)
df["ALT_len"] = df["ALT"].apply(len)

Load RF Model + Predict Pathogenicity

In [19]:
MODEL = MODEL_DIR + "rf_ann_model.pkl"
model = joblib.load(MODEL)

features = df[
    ["CHROM", "POS", "REF_len", "ALT_len", "Effect", "Impact", "Transcript_BioType"]
]

df["Prediction"]  = model.predict(features)
df["Probability"] = model.predict_proba(features)[:,1]

Save Final CSV

In [20]:
OUTPUT_CSV = VARIANT_DIR + "patient_final_variants.csv"
df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

Saved: /content/drive/MyDrive/Bioinformatics/outputs/variants/patient_final_variants.csv


Extract Top 20 Most Pathogenic Variants

In [22]:
top20 = df.sort_values(by="Probability", ascending=False).head(20)

top20_cols = [
    "CHROM", "POS", "REF", "ALT",
    "GeneName", "Effect", "Impact",
    "Probability", "Prediction"
]

top20_display = top20[top20_cols]
top20_display

,CHROM,POS,REF,ALT,GeneName,Effect,Impact,Probability,Prediction
642518,chr13,32338844,TACTC,T,BRCA2,frameshift_variant,HIGH,0.735049,1
642516,chr13,32337218,CT,C,BRCA2,frameshift_variant,HIGH,0.733739,1
642519,chr13,32339569,CT,C,BRCA2,frameshift_variant,HIGH,0.733711,1
642483,chr13,32202452,AAGAC,A,FRY,frameshift_variant,HIGH,0.733393,1
604452,chr12,31792140,ATGAG,A,H3F3C,frameshift_variant,HIGH,0.732964,1
604449,chr12,31791815,TA,T,H3F3C,frameshift_variant,HIGH,0.732760,1
867906,chr22,31956246,GA,G,YWHAH,frameshift_variant,HIGH,0.732760,1
642475,chr13,32179694,TC,T,FRY,frameshift_variant,HIGH,0.732652,1
867974,chr22,32224438,CA,C,SLC5A4,frameshift_variant,HIGH,0.732652,1
642467,chr13,32173474,AC,A,FRY,frameshift_variant,HIGH,0.732652,1


Save Top 20 to Drive

In [23]:
TOP20_CSV = "/content/drive/MyDrive/Bioinformatics/outputs/variants/patient1_top20_pathogenic.csv"
top20_display.to_csv(TOP20_CSV, index=False)

print("Saved Top 20 pathogenic variants:", TOP20_CSV)

Saved Top 20 pathogenic variants: /content/drive/MyDrive/Bioinformatics/outputs/variants/patient1_top20_pathogenic.csv
